In [53]:
# Transformers example -> Reverseing sequence of numbers


In [125]:
import jax
import jax.numpy as jnp
import jax.random as jrandom

import haiku as hk
import optax

from probjax.nn.transformers import Transformer, embeding, LearnedPosEmbed, PosEmbed

In [126]:
VOCAB_SIZE = 10

In [127]:
def generate_data(key, n, T, vocab_size=10):
    sequences = jrandom.randint(key, (n, T,1), 0, vocab_size, dtype=jnp.int32)
    # Just reversing
    # sequences_reversed = jnp.array(jnp.flip(sequences, axis=-2), dtype=jnp.int32)
    # Sorting
    sequences_reversed = jnp.sort(sequences, axis=-2)

    
    return sequences, sequences_reversed


inputs, labels = generate_data(jrandom.PRNGKey(0), 100, 20)

In [128]:
key = jrandom.PRNGKey(0)

In [134]:
@hk.transform
def f(x):
    # Embed
    x = jnp.squeeze(hk.Embed(VOCAB_SIZE, 20)(x), -2)
    # x = jnp.squeeze(jax.nn.one_hot(x, VOCAB_SIZE))
    x = PosEmbed(x.shape[-1],max_len=500)(x)
    # Encode
    model = Transformer(num_heads=1, num_layers=1, attn_size=x.shape[-1], dropout_rate=0.0)
    embedding = model(x)
    logits = hk.Linear(VOCAB_SIZE)(embedding)
    return logits


In [135]:
params = f.init(key, inputs)
outputs = f.apply(params, key, inputs)

In [136]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

In [137]:

def loss_fn(params, inputs, outputs, rng):
    inp_data, labels = inputs, outputs
    logits = f.apply(params, rng, inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    loss = optax.softmax_cross_entropy(logits, labels).mean()
    return loss

def acc(params, inputs, outputs, rng):
    inp_data, labels = inputs, outputs
    logits = f.apply(params, rng, inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params, inputs, outputs, rng, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params, inputs, outputs, rng)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

In [139]:
train_seq_len = [10, 50, 100]
for i in range(1000):
    key, subkey, key2 = jrandom.split(jrandom.PRNGKey(i), 3)
    inputs, labels = generate_data(key, 512, train_seq_len[i%len(train_seq_len)], vocab_size=VOCAB_SIZE)
    loss, params, opt_state = update(params, inputs, labels, subkey, opt_state)
    if (i % 100) == 0:
        accuracy = acc(params, inputs, labels, key2)
        print(accuracy, loss)

0.82128906 0.55851364
0.77722657 0.5617341
0.7792773 0.51980704
0.9021484 0.3863251
0.82609373 0.44574478
0.80615234 0.46013156
0.9482422 0.26524538
0.85734373 0.36512297
0.8388281 0.39211103
0.96015626 0.2070519


In [150]:
outputs = f.apply(params, key, jax.random.randint(key, (1, 100,1), 0, 10, dtype=jnp.int32))

In [151]:
outputs.argmax(-1)

Array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8,
        9, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]], dtype=int32)